In [1]:
import pymupdf4llm
loaded_document=pymupdf4llm.to_markdown("edited structured tables.pdf")


Processing edited structured tables.pdf...
[                                        ] (0/13[                                        ] (  1/13[                                        ] (  2/13[                                        ] (  3/138[=                                       ] (  4/138[=                                       ] (  5/138[=                                       ] (  6/13[==                                      ] (  7/13[==                                      ] (  8/13[==                                      ] (  9/13[==                                      ] ( 10/138[===                                     ] ( 11/1[===                                     ] ( 12/1[===                                     ] ( 13/13[====                                    ] ( 14/13[====                                    ] ( 15/13[====                                    ] ( 16/13[====                                    ] ( 17/138[=====                                   ] ( 18/138[====

In [2]:
with open("man2.md","w",encoding="utf-8") as f:
    f.write(loaded_document)


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings()
# embeddings = HuggingFaceEmbeddings(model_name="nomic-ai/nomic-embed-text-v1.5",trust_remote_code=True)


In [16]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.storage._lc_store import create_kv_docstore
from langchain.storage import LocalFileStore
from langchain_chroma import Chroma

#vector_store
vector_store = Chroma(collection_name="HR_manual",embedding_function=embeddings,persist_directory="./RAG")
# create local_store
fs = LocalFileStore("./store_location")
store = create_kv_docstore(fs)


# define child splitter
child_splitter=RecursiveCharacterTextSplitter(
    chunk_size=200,  # must be less
    chunk_overlap=30,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
    # length_function=len
)

# Define parent splitter
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=4000)


In [17]:
from langchain.retrievers import ParentDocumentRetriever
retriever = ParentDocumentRetriever(
    vectorstore=vector_store,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    # search_kwargs = {"k": 20},
)

In [5]:
len(list(store.yield_keys()))

0

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os
load_dotenv()
api_key=os.getenv("GOOGLE_API_KEY")
llm=ChatGoogleGenerativeAI(model="gemini-1.5-flash",api_key=api_key)

In [7]:
# loaded_document
from langchain_community.document_loaders import UnstructuredMarkdownLoader
loader = UnstructuredMarkdownLoader("man2.md")
data = loader.load()


In [10]:
retriever.add_documents(data)

In [11]:
len(list(store.yield_keys()))

198

In [26]:
ex=llm.invoke(f"you are assistant that is use for query expansion. you need to apply expansion query to the given query and return a single expanded query. this is the query:  mobile policy")

In [27]:
ex

AIMessage(content='mobile device policy', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-70ca197d-82ad-4764-a0ee-33f695086d91-0', usage_metadata={'input_tokens': 35, 'output_tokens': 4, 'total_tokens': 39, 'input_token_details': {'cache_read': 0}})

In [30]:
query="maximum age required by grade 1 officer?"
result=retriever.invoke("what policies are define about mobiles in manual")
# result=vector_store.similarity_search(query.lower())
print(result)
llm.invoke(f"you are assistant that tell that the context given to you provide accurate answer. just reply with yes or no. this in context:{result} and this is query {query}").content


[Document(metadata={'source': 'man2.md'}, page_content="79.13 The infringement of any of the above Rules may render the allottee liable to cancellation of the accommodation allotted to him.\n\nMobile Phone Policy\n\n80.00 Statement of Intent\n\n80.1 Due to the exigencies of business, SNGPL requires certain members of management to remain accessible by colleagues and customers when out of the office. Mobile phones are therefore provided to such employees to enable them to remain in touch and thereby promote the smooth functioning of the Company's business.\n\n80.2 The paramount consideration when determining eligibility for a mobile telephone facility is the need for accessibility of the employee. It is important to remember that this facility is assigned strictly on this basis and is NOT to be considered a perk of the position.\n\n81.00 Eligibility\n\n81.1 All executives in grades VIII - X are entitled to the facility of a mobile phone. In\n\naddition to senior management, the Company 

'No'

In [33]:
query="mobile policy"
result=retriever.invoke(query.lower())
len(result[1].page_content)

1699

In [17]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from langchain_community.document_loaders import UnstructuredMarkdownLoader

# ✅ Download necessary NLTK resources
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")

def preprocess_text(text):
    """Cleans and preprocesses text before further use."""
    tokens = word_tokenize(text.lower())  # Lowercasing & tokenization
    tokens = [word for word in tokens if word.isalnum()]  # Remove punctuation
    tokens = [word for word in tokens if word not in stopwords.words("english")]  # Remove stopwords
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]  # Lemmatization
    return " ".join(tokens)

# ✅ Load raw document
loader = UnstructuredMarkdownLoader("man.md")
data = loader.load()

# ✅ Extract and preprocess text
raw_texts = [doc.page_content for doc in data]
preprocessed_texts = [preprocess_text(text) for text in raw_texts]

# ✅ Output preprocessed text (for debugging)
for i, text in enumerate(preprocessed_texts):
    print(f"Processed Document {i+1}:")
    print(text[:500])  # Print first 500 chars of each processed document
    print("-" * 80)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\husnain.mahmood\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\husnain.mahmood\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\husnain.mahmood\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Processed Document 1:
executive summary executive service rule first introduced may 1981 amended per bod approval time time currently applicable direction hr r nc bod service hr consultant commissioned review assess suitability existing rule per current market condition propose amendment policy practice document prepare new manual financial admin power position level also made integral part hr manual per advice chairman hr r nc board director approved hr manual 344th meeting held addition amendment different clause t
--------------------------------------------------------------------------------
